# 04 — Kustlinjeförändring: Ystad, Skåne (2018–2025)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/space-data-lab/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_050-Coastline-Ystad.ipynb)

**Syfte:** Kartlägga årlig förändring av Ystads kustlinje 2018–2025 för att dokumentera kusterosion och sedimenttransport. Klimatanpassning: erosion är ett av Sveriges mest synliga och kostsamma klimateffekter längs södra och östra kusten.

**Partners som bidragit:**
- **Statens Geotekniska Institut (SGI)** — erosionsövervakning, geotekniska referensdata
- **SMHI** — havsvattenstånd, vågdata
- **Stockholms universitet (Inst. för naturgeografi)** — kustmorfologisk expertis
- **Länsstyrelsen Skåne** — användare av resultat
- **RISE** — ImintEngine, CoastSat-implementation

**Datakällor:**
- Copernicus Sentinel-2 L2A — 8 sommardatum (2018–2025) via Digital Earth Sweden
- CoastSat-metod (Vos et al., 2019) för NDWI/MNDWI + Otsu-tröskling
- CoastSeg SegFormer (tränad på satellitbilder) för semantic segmentation

**Licens:** CC0 1.0 (notebook) · CoastSeg GPL-3.0

## 1. Metod

Två parallella metoder för kustlinjeextraktion:

| Analyzer | Metod | Princip |
|----------|-------|---------|
| `shoreline` (index) | NDWI + MNDWI + Otsu | Automatisk tröskel på vattenindex — snabbt, robust |
| `shoreline` (segformer) | CoastSeg SegFormer (4-class) | Djupinlärning: water/whitewater/sand/other |

**Per-år-analys:**
1. Hämta ett molnfritt Sentinel-2-datum från varje sommar (juni–aug)
2. Beräkna NDWI = (Green − NIR) / (Green + NIR)
3. Beräkna MNDWI = (Green − SWIR) / (Green + SWIR)
4. Otsu-tröskling → binär vatten/land-mask
5. Vektorisera konturen → kustlinje som GeoJSON

**Resultat från showcase-AOI (Ystad 2018–2025):**
- 8 årliga kustlinjer extraherade
- Water fraction varierar mellan 60.5% (2018) och 61.9% (2025) → 1.25% ökning på 7 år
- Svag trend mot mer vatten (erosion) men inom mätbrus

## 2. Setup

In [ ]:
from executors.local import LocalExecutor
import matplotlib.pyplot as plt
import folium

# Ystads kustavsnitt — från showcase-metadata
AOI = {
    "west": 14.13,
    "south": 55.355,
    "east": 14.22,
    "north": 55.400,
}

# 8 sommardatum (2018–2025) — ett per år
YEARS = {
    2018: "2018-06-03",
    2019: "2019-08-27",
    2020: "2020-08-14",
    2021: "2021-06-17",
    2022: "2022-06-25",
    2023: "2023-06-07",
    2024: "2024-08-10",
    2025: "2025-06-14",
}

print(f"AOI (Ystad): {AOI}")
print(f"Analyserade år: {list(YEARS.keys())}")

## 3. Kör analys för alla 8 år

In [ ]:
executor = LocalExecutor(
    output_dir="outputs/kustlinje_ystad",
    config_path="config/analyzers.yaml",
)

yearly_results = {}
for year, date in YEARS.items():
    print(f"Analyserar {year} ({date})...")
    r = executor.execute(date=date, coords=AOI)
    yearly_results[year] = r

print(f"\nKlart — {len(yearly_results)} år analyserade.")

## 4. Visualisering

In [ ]:
# Kartan — alla kustlinjer lagrade över varandra
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=13, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellit",
).add_to(m)

# Färgskala: röd (2018) → blå (2025)
import matplotlib.colors as mcolors
cmap = plt.cm.RdYlBu
years_sorted = sorted(yearly_results.keys())

for i, year in enumerate(years_sorted):
    r = yearly_results[year]
    if "shoreline" in r.analyses:
        vectors = r.analyses["shoreline"].data.get("vectors", [])
        color = mcolors.to_hex(cmap(i / (len(years_sorted) - 1)))
        folium.GeoJson(
            vectors,
            style_function=lambda x, c=color: {"color": c, "weight": 2},
            name=f"{year}",
        ).add_to(m)

folium.LayerControl().add_to(m)
m

In [ ]:
# Trendanalys: water fraction per år
water_fractions = {}
for year, r in yearly_results.items():
    if "shoreline" in r.analyses:
        wf = r.analyses["shoreline"].data.get("water_fraction")
        if wf is not None:
            water_fractions[year] = wf

fig, ax = plt.subplots(figsize=(10, 5))
years = sorted(water_fractions.keys())
values = [water_fractions[y] for y in years]
ax.plot(years, values, "o-", color="#1f77b4", markersize=10)
ax.set_xlabel("År")
ax.set_ylabel("Vattenfraktion i AOI")
ax.set_title("Ystad kustlinje — trendanalys 2018–2025")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if len(years) >= 2:
    delta = values[-1] - values[0]
    print(f"Förändring {years[0]} → {years[-1]}: {delta:+.4f} ({delta*100:+.2f} procentenheter)")

## 5. Tolkning

**Vad 8 års tidsserie visar:**
- Trend på ~1.25 procentenheter ökad vattenfraktion över 7 år
- Inom AOI (~5 × 5 km) motsvarar detta en förskjutning av kustlinjen på meternivå
- Sommardatum valda för minimal vegetationsstörning och låg havsnivå

**Begränsningar:**
- 10 m-pixlar → minsta detekterbara förändring = ~5 m (sub-pixel med CoastSat)
- Vind/vågor på bilddatumet kan skapa falsk "erosion" → kräver vågkontext från SMHI
- Strandtyp påverkar signaturen — sandstränder tydligast, klippstränder svårare

**Samhällsnytta:**
- Ystads kommun + Länsstyrelsen Skåne: underlag för strandfodring och kustskydd
- SGI: komplement till TERRAIN-fältmätningar
- Klimatanpassning: del av svensk kustnära riskbedömning (ref Boverket, MSB)

**Nästa steg:**
- Utöka till hela Skånes sydkust (Åhus → Trelleborg)
- Koppla till SMHI:s havsvattenstånd för rätt referensnivå
- Validera mot SGI:s TERRAIN-mätningar
- Lägg till stormeventsdetektion (före/efter)

### Länkar
- [ImintEngine kustlinje-showcase](https://digitalearth.se/case/)
- [ImintEngine/imint/analyzers/shoreline.py](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/shoreline.py)
- [CoastSat (Vos et al., 2019)](https://github.com/kvos/CoastSat)
- [SGI:s kusterosionskartor](https://gis2.sgi.se)